# FerrisRes — teacher benchmark on Kaggle 2×T4

Measures **prefill tokens/s at the training sequence length**, which is the
cost that matters: a teacher is run forward over the corpus, not sampled
from. The output is `metric=teacher_prefill_tps value=<x>` and a table of
tokens/s and hours per 10M tokens.

Prerequisites:
- GPU T4 ×2, internet enabled.
- A `ferrisres_*_amd64.deb` built from a branch that has `devices`,
  `benchmark teacher` and the `qwen3-0.6b` preset, attached as a Kaggle
  dataset at `/kaggle/input/ferrisres-deb/`.
- Quota: this notebook is minutes, not hours.

In [ ]:
from pathlib import Path
    import json, os, re, subprocess

    DEB_DIR = Path("/kaggle/input/ferrisres-deb")
    MODELS = {
        "qwen3-0.6b-q8": ("Qwen/Qwen3-0.6B-GGUF", "Qwen3-0.6B-Q8_0.gguf", "qwen3-0.6b"),
        "qwen3-0.6b-q4": ("ertghiu256/qwen3-0.6b-instruction-following-gguf", "Qwen3-0.6B.Q4_K_M.gguf", "qwen3-0.6b"),
    }
    SEQ_LENS = [512, 2048]
    ITERATIONS = 3
    WORK = Path("/kaggle/working/teacher"); WORK.mkdir(parents=True, exist_ok=True)

In [ ]:
!nvidia-smi -L
    !ls /usr/share/vulkan/icd.d 2>/dev/null | head
    deb = sorted(DEB_DIR.glob("ferrisres_*_amd64.deb"))
    assert deb, f"no .deb in {DEB_DIR}; attach a dataset with the built package"
    !dpkg -i {deb[-1]}
    !ferrisres devices

In [ ]:
# Download GGUF + tokenizer if internet is on. Skip if the files are attached.
    from huggingface_hub import hf_hub_download
    paths = {}
    for name, (repo, filename, config) in MODELS.items():
        try:
            p = hf_hub_download(repo_id=repo, filename=filename, local_dir=WORK / name)
            paths[name] = Path(p)
        except Exception as e:
            print(name, "download failed:", e)
    tok = hf_hub_download(repo_id="Qwen/Qwen3-0.6B", filename="tokenizer.json", local_dir=WORK / "tok")
    print(json.dumps({k: str(v) for k, v in paths.items()}, indent=2))

In [ ]:
rows = []
    for name, path in paths.items():
        config = MODELS[name][2]
        for seq in SEQ_LENS:
            cmd = [
                "ferrisres", "benchmark", "teacher",
                "--model-path", str(path), "--model-format", "gguf",
                "--config", config, "--tokenizer", str(tok),
                "--seq-len", str(seq), "--iterations", str(ITERATIONS),
                "--warmup", "1", "--json",
            ]
            print("$", " ".join(cmd))
            out = subprocess.run(cmd, text=True, capture_output=True)
            print(out.stdout[-400:], out.stderr[-200:])
            m = re.search(r"metric=teacher_prefill_tps value=([0-9.]+)", out.stdout)
            tps = float(m.group(1)) if m else None
            rows.append({"model": name, "seq": seq, "tokens_per_s": tps,
                         "hours_per_10M": round(1e7 / (tps * 3600), 1) if tps else None})
    print(json.dumps(rows, indent=2))

## Interpret

- Compare `hours_per_10M` across teachers; that is the distillation budget.
- Record the device: a CPU fallback is not a T4 result.
- Q4 is not Q8; the quant stays in the row.
- This notebook does not measure teacher quality. Run `evaluate` on the same
  held-out text for that.